# Aula 2 — Transformações Avançadas com Spark

**Disciplina:** Big Data Processing — MBA Engenharia de Dados (Mackenzie)  
**Professor:** Alexandre Tavares  

---

## Cenário de Negócio

A **DataFlow Analytics** cresceu 10x nos últimos 6 meses. A Product Owner (Ana) precisa de relatórios para a **Black Friday** que cruzam:

- **Vendas** (1 milhão de registros) — tabela de fatos
- **Clientes** (500 mil registros) — tabela de dimensão grande
- **Categorias** (10 registros) — tabela de dimensão pequena

Com pandas, a tentativa de `merge()` resultou em `MemoryError`. A solução: **joins distribuídos do Spark**.

---

## O Que Você Vai Aprender

1. **Joins** — como cruzar DataFrames (inner, left, broadcast, anti)
2. **Window Functions** — rankings e tendências sem perder linhas
3. **UDFs** — funções customizadas e seu impacto na performance
4. **Plano de Execução** — como o Spark realmente executa sua query

---

## Instruções

1. Execute cada célula na ordem (Shift+Enter)
2. **Leia os comentários** — cada linha está explicada
3. Observe os resultados e compare com o esperado
4. Ao final, resolva o **Desafio**

---

---

## 1. Configuração — Criar a SparkSession

A **SparkSession** é o ponto de entrada para toda interação com o Spark.  
Aqui configuramos parâmetros importantes para o lab:

| Parâmetro | Valor | Por quê |
|-----------|-------|--------|
| `appName` | DataFlow-Aula02 | Identificar no Spark UI |
| `master` | local[*] | Usar todos os cores da máquina |
| `driver.memory` | 2g | Memória para joins em modo local |
| `shuffle.partitions` | 8 | Menos partições = menos overhead (local) |
| `autoBroadcastJoinThreshold` | 10m | Tabelas < 10MB são broadcast automaticamente |

In [ ]:
# =============================================================================
# PASSO 1: Criar a SparkSession (ponto de entrada do Spark)
# =============================================================================

from pyspark.sql import SparkSession  # Classe principal para interagir com Spark

# Construir a sessão com configurações otimizadas para o lab
spark = SparkSession.builder \
    .appName("DataFlow-Aula02") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .getOrCreate()

# Confirmar que a sessão foi criada com sucesso
print(f"✅ SparkSession criada com sucesso!")
print(f"   Versão Spark: {spark.version}")
print(f"   App Name:     {spark.sparkContext.appName}")
print(f"   Master:       {spark.sparkContext.master}")
print(f"   Shuffle Parts: {spark.conf.get('spark.sql.shuffle.partitions')}")

---

## 2. Carga de Dados — 3 Fontes Diferentes

Vamos carregar os 3 datasets que representam as fontes da DataFlow:

| Fonte | Formato | Registros | Descrição |
|-------|---------|-----------|----------|
| Vendas | Parquet | ~1.000.000 | Tabela de fatos — cada venda realizada |
| Clientes | Parquet | ~500.000 | Dimensão — dados cadastrais dos clientes |
| Categorias | JSON | 10 | Dimensão pequena — categorias de produtos |

### Por que formatos diferentes?

- **Parquet** → formato colunar, comprimido, otimizado para leitura analítica
- **JSON** → formato comum de APIs e sistemas legados

Na vida real, dados vêm de fontes heterogêneas. O Spark lê todos nativamente.

In [ ]:
# =============================================================================
# PASSO 2: Carregar os 3 datasets
# =============================================================================

from pyspark.sql.functions import explode, col  # explode: transforma array em linhas

# --- FONTE 1: Vendas (Parquet, 1M registros) ---
# Parquet é colunar — Spark lê apenas as colunas necessárias (eficiente!)
df_vendas = spark.read.parquet("/home/jovyan/work/data/aula_02/vendas_2023_completo.parquet")

# --- FONTE 2: Clientes (Parquet, 500K registros) ---
df_clientes = spark.read.parquet("/home/jovyan/work/data/aula_02/clientes.parquet")

# --- FONTE 3: Categorias (JSON aninhado, 10 registros) ---
# O JSON tem estrutura: {"categorias": [{...}, {...}, ...]}
# Precisamos "explodir" o array para transformar em tabela plana
df_categorias_raw = spark.read.json(
    "/home/jovyan/work/data/aula_02/categorias.json",
    multiLine=True  # JSON é um objeto único (não JSON Lines)
)

# explode() transforma cada elemento do array "categorias" em uma linha separada
# Depois selecionamos apenas category_id e category_name
df_categorias = df_categorias_raw \
    .select(explode(col("categorias")).alias("cat")) \
    .select(
        col("cat.category_id").alias("category_id"),      # Ex: "CAT_01"
        col("cat.category_name").alias("category_name")   # Ex: "Eletrônicos"
    )

# --- Verificar contagens ---
print("📊 Datasets carregados:")
print(f"   Vendas:     {df_vendas.count():>10,} registros | {len(df_vendas.columns)} colunas")
print(f"   Clientes:   {df_clientes.count():>10,} registros | {len(df_clientes.columns)} colunas")
print(f"   Categorias: {df_categorias.count():>10,} registros | {len(df_categorias.columns)} colunas")
print()
print("📋 Tabela de Categorias (completa):")
df_categorias.show(truncate=False)

---

### 2.1 Explorar os Schemas

Antes de cruzar tabelas, precisamos entender a **estrutura** de cada uma:  
- Quais colunas existem?
- Quais são as **chaves** de ligação entre as tabelas?
- Quais tipos de dados?

**Chaves de ligação:**
- Vendas ↔ Clientes: `customer_id`
- Vendas ↔ Categorias: `product_id` → derivar `category_id`

In [ ]:
# =============================================================================
# PASSO 2.1: Explorar estrutura dos DataFrames
# =============================================================================

# printSchema() mostra nome da coluna, tipo e se aceita null
print("═" * 50)
print("SCHEMA: VENDAS (tabela de fatos)")
print("═" * 50)
df_vendas.printSchema()

print("═" * 50)
print("SCHEMA: CLIENTES (dimensão grande)")
print("═" * 50)
df_clientes.printSchema()

print("═" * 50)
print("SCHEMA: CATEGORIAS (dimensão pequena)")
print("═" * 50)
df_categorias.printSchema()

# Amostra de vendas para entender os dados
print("\n📋 Amostra de Vendas (5 primeiros registros):")
df_vendas.show(5, truncate=False)

---

## 3. Joins — Cruzando Fontes de Dados

### O que é um Join?

Um **join** combina registros de dois DataFrames baseado em uma coluna em comum (chave).  
É o equivalente do `VLOOKUP` do Excel, mas distribuído e para milhões de registros.

### Estratégia para a DataFlow:

```
df_vendas (1M)  ──LEFT JOIN──►  df_clientes (500K)  ──BROADCAST JOIN──►  df_categorias (10)
     chave: customer_id                 chave: category_id (derivada)
```

### Tipos de Join que usaremos:

| Tipo | O que faz | Quando usar |
|------|----------|-------------|
| `left` | Mantém TODAS as vendas, enriquece com dados de cliente | Não queremos perder vendas |
| `broadcast` | Envia tabela pequena para todos os executors (sem shuffle) | Categorias tem só 10 linhas |

### 3.1 LEFT JOIN: Vendas + Clientes

**Por que LEFT JOIN?**  
Queremos TODAS as vendas no relatório — mesmo que alguma venda referencie um cliente que não existe na tabela de clientes (guest checkout, por exemplo).

- Se usássemos `inner`: perderíamos vendas de clientes não-cadastrados
- Com `left`: mantemos 100% das vendas, campos de clientes ficam `null` quando não há match

In [ ]:
# =============================================================================
# PASSO 3.1: LEFT JOIN — Vendas + Clientes
# =============================================================================

from pyspark.sql.functions import (
    col,             # Referência a coluna
    broadcast,       # Instrui Spark a enviar tabela pequena para todos os workers
    coalesce,        # Retorna o primeiro valor não-null entre os argumentos
    lit,             # Cria coluna com valor literal (constante)
    regexp_extract,  # Extrai substring usando regex
    concat,          # Concatena strings
    lpad,            # Padding à esquerda (ex: "1" → "01")
    ceil as spark_ceil  # Arredonda para cima (renomeado para não conflitar)
)

# LEFT JOIN: vendas (esquerda) + clientes (direita)
# on="customer_id" → coluna de ligação (existe em ambas as tabelas)
# how="left"       → mantém TODOS os registros da tabela esquerda (vendas)
df_com_clientes = df_vendas.join(
    df_clientes,          # DataFrame da direita
    on="customer_id",     # Coluna de junção (deve ter o mesmo nome em ambos)
    how="left"            # Tipo: preserva todas as vendas
)

# --- Verificar resultado ---
count_vendas = df_vendas.count()
count_pos_join = df_com_clientes.count()

print("📊 Resultado do LEFT JOIN (vendas + clientes):")
print(f"   Vendas originais:    {count_vendas:>10,}")
print(f"   Após left join:      {count_pos_join:>10,}")
print(f"   Preservou 100%?      {'✅ SIM' if count_vendas == count_pos_join else '❌ NÃO'}")
print()

# Mostrar colunas disponíveis após o join
print(f"   Colunas totais: {len(df_com_clientes.columns)}")
print(f"   Lista: {sorted(df_com_clientes.columns)}")

### 3.2 Derivar category_id a partir do product_id

A tabela de vendas tem `product_id` (ex: `PROD_0456`), mas **não tem** `category_id` diretamente.

Precisamos derivar a categoria usando a regra de negócio:  
- Produtos `PROD_0001` a `PROD_0500` → Categoria `CAT_01`  
- Produtos `PROD_0501` a `PROD_1000` → Categoria `CAT_02`  
- E assim por diante (500 produtos por categoria, 10 categorias = 5000 produtos)

**Fórmula:** `category_id = "CAT_" + ceil(numero_produto / 500)`

In [ ]:
# =============================================================================
# PASSO 3.2: Derivar category_id do product_id
# =============================================================================

# Lógica:
# 1. regexp_extract → extrai o número do product_id ("PROD_0456" → "0456")
# 2. cast("int")   → converte string "0456" para inteiro 456
# 3. / 500         → divide por 500 (grupo de categoria)
# 4. spark_ceil()  → arredonda para cima (456/500 = 0.912 → 1)
# 5. lpad(2, "0") → garante 2 dígitos ("1" → "01")
# 6. concat("CAT_", ...) → monta o ID final ("CAT_01")

df_com_clientes = df_com_clientes.withColumn(
    "category_id",                                          # Nome da nova coluna
    concat(
        lit("CAT_"),                                        # Prefixo fixo
        lpad(
            spark_ceil(
                regexp_extract(                              # Extrair número do PROD_XXXX
                    col("product_id"),                       # Coluna de origem
                    r"PROD_(\d+)",                          # Regex: captura dígitos após PROD_
                    1                                        # Grupo de captura 1
                ).cast("int")                                # String → inteiro
                / 500                                        # Dividir por 500 (produtos por categoria)
            ).cast("int").cast("string"),                    # Resultado → inteiro → string
            2, "0"                                           # Pad left com "0" até 2 dígitos
        )
    )
)

# --- Verificar a derivação ---
print("📋 Verificação da derivação product_id → category_id:")
df_com_clientes.select("product_id", "category_id") \
    .distinct() \
    .orderBy("product_id") \
    .show(10, truncate=False)

### 3.3 BROADCAST JOIN: Enriquecer com Categorias

**O que é Broadcast Join?**

Quando uma tabela é **muito pequena** (< 10MB), o Spark pode copiar ela inteira para todos os workers.  
Isso elimina o **shuffle** (redistribuição de dados pela rede) — operação muito cara.

```
SEM broadcast: 1M registros precisam ser redistribuídos pela rede (LENTO)
COM broadcast: tabela de 10 linhas é copiada para cada worker (RÁPIDO)
```

A tabela de categorias tem apenas **10 linhas** — caso perfeito para broadcast!

In [ ]:
# =============================================================================
# PASSO 3.3: BROADCAST JOIN — Vendas + Categorias (tabela pequena)
# =============================================================================

# broadcast(df_categorias) → instrui o Spark a copiar esta tabela
#                            para todos os executors (sem shuffle!)
df_completo = df_com_clientes.join(
    broadcast(df_categorias),    # Broadcast: copia 10 linhas para todos os workers
    on="category_id",            # Chave de junção (derivada no passo anterior)
    how="left"                   # Mantém todas as vendas mesmo sem categoria
)

# --- Tratar nulls ---
# coalesce(A, B) retorna A se não for null, senão retorna B
# Vendas sem match na tabela de categorias terão category_name = null
# Substituímos por "Sem Categoria" para o relatório ficar limpo
df_completo = df_completo.withColumn(
    "category_name",
    coalesce(col("category_name"), lit("Sem Categoria"))
)

# --- Verificar resultado ---
print(f"📊 DataFrame completo (vendas + clientes + categorias): {df_completo.count():,} registros")
print(f"   Colunas: {len(df_completo.columns)}")
print()
print("📋 Amostra do resultado enriquecido:")
df_completo.select(
    "order_id", "customer_name", "category_name", "total_amount", "shipping_state"
).show(10, truncate=20)

### 3.4 Verificar o Plano de Execução

O `explain()` mostra COMO o Spark vai executar a query:  
- `BroadcastHashJoin` = broadcast funcionou ✅
- `SortMergeJoin` = join normal com shuffle ⚠️
- `Filter` antes do `Scan` = predicate pushdown ✅

In [ ]:
# =============================================================================
# PASSO 3.4: Verificar plano de execução (explain)
# =============================================================================

# explain(True) mostra plano lógico + plano físico
# Procure por:
#   - "BroadcastHashJoin" → confirma que categorias usou broadcast
#   - "SortMergeJoin"     → join de vendas com clientes (ambas grandes)
print("═" * 60)
print("PLANO DE EXECUÇÃO DO PIPELINE COMPLETO")
print("═" * 60)
df_completo.explain(True)

### 3.5 Comparar: COM Broadcast vs SEM Broadcast

Para visualizar a diferença, vamos comparar os planos de execução  
do mesmo join **com** e **sem** a dica de broadcast.

In [ ]:
# =============================================================================
# PASSO 3.5: Comparar planos COM vs SEM broadcast
# =============================================================================

# --- COM broadcast (otimizado) ---
# Esperado: BroadcastHashJoin (sem shuffle da tabela de 1M)
print("═" * 60)
print("COM BROADCAST — Tabela pequena copiada para cada worker")
print("═" * 60)
df_com_clientes.join(
    broadcast(df_categorias),  # Forçar broadcast
    on="category_id",
    how="left"
).explain()

print()
print("═" * 60)
print("SEM BROADCAST — Ambas tabelas redistribuídas (shuffle)")
print("═" * 60)
# Desabilitar auto-broadcast temporariamente para forçar SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

df_com_clientes.join(
    df_categorias,             # Sem broadcast
    on="category_id",
    how="left"
).explain()

# Restaurar configuração original
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")

print()
print("💡 Observe: COM broadcast não tem 'Exchange' (shuffle).")
print("   SEM broadcast tem 'Exchange hashpartitioning' — redistribui dados pela rede.")

---

## 4. Window Functions — Rankings e Tendências

### O que são Window Functions?

Permitem calcular valores **relativos a um grupo** sem reduzir o número de linhas.

| Operação | Comportamento | Resultado |
|----------|--------------|----------|
| `groupBy().agg()` | **Colapsa** linhas em grupos | 1 linha por grupo |
| **Window Function** | Calcula sobre uma "janela" | **Mantém todas** as linhas |

### Analogia Excel:
- `groupBy` = Tabela Dinâmica (pivot) — reduz para N linhas
- `Window` = Fórmula que olha linhas vizinhas — mantém todas as linhas

### Componentes de uma Window:
```python
Window
    .partitionBy("grupo")   # Como dividir (similar ao GROUP BY)
    .orderBy("ordem")       # Como ordenar dentro de cada grupo
```

### 4.1 Ranking: Top Clientes por Estado

A Ana quer: **"Quem são os top 3 clientes por faturamento em cada estado?"**

Para isso usamos `dense_rank()` — atribui posição dentro de cada grupo (estado),  
ordenado por faturamento total (maior primeiro).

In [ ]:
# =============================================================================
# PASSO 4.1: Ranking — Top clientes por faturamento em cada estado
# =============================================================================

from pyspark.sql.functions import sum, count, avg, round, desc  # Funções de agregação
from pyspark.sql.window import Window                           # Construtor de janelas
from pyspark.sql.functions import dense_rank, row_number        # Funções de ranking
from pyspark.sql.functions import lag, lead                     # Acesso a linhas vizinhas

# --- Passo A: Agregar faturamento por cliente+estado ---
# Primeiro precisamos saber quanto cada cliente gastou no total
df_faturamento_cliente = df_completo \
    .groupBy("shipping_state", "customer_id") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento_total"),  # Soma dos valores
        count("order_id").alias("num_pedidos")                     # Qtd de pedidos
    )

# --- Passo B: Definir a janela de ranking ---
# partitionBy("shipping_state") → ranking SEPARADO para cada estado
# orderBy(desc(...))            → do maior faturamento para o menor
window_ranking = Window \
    .partitionBy("shipping_state") \
    .orderBy(desc("faturamento_total"))

# --- Passo C: Aplicar dense_rank() ---
# dense_rank(): 1, 2, 3, 4... (sem pular posições em caso de empate)
# row_number(): 1, 2, 3, 4... (sempre único, desempata arbitrariamente)
# rank():       1, 1, 3, 4... (pula posição após empate)
df_ranking = df_faturamento_cliente.withColumn(
    "posicao",                           # Nome da nova coluna
    dense_rank().over(window_ranking)     # Aplica ranking dentro da janela
)

# --- Passo D: Filtrar apenas top 3 por estado ---
print("🏆 TOP 3 CLIENTES POR ESTADO (por faturamento):")
print()
df_ranking \
    .filter(col("posicao") <= 3) \
    .orderBy("shipping_state", "posicao") \
    .show(15, truncate=False)

### 4.2 Tendência Temporal: lag() e lead()

A Ana quer: **"Cada cliente está gastando mais ou menos ao longo do tempo?"**

- `lag(coluna, 1)` → acessa o valor da **linha anterior** (compra passada)
- `lead(coluna, 1)` → acessa o valor da **próxima linha** (compra futura)

Com isso calculamos a **variação** entre compras consecutivas de cada cliente.

In [ ]:
# =============================================================================
# PASSO 4.2: Tendência — Comparar cada compra com a anterior (lag)
# =============================================================================

# Janela: para cada cliente, ordenar por data de compra (cronológico)
window_temporal = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date")

# Criar DataFrame com valor anterior e variação
df_tendencia = df_completo \
    .select("customer_id", "order_date", "total_amount") \
    .withColumn(
        "compra_anterior",                            # Nova coluna
        lag("total_amount", 1).over(window_temporal)  # Valor da compra anterior
        # lag(coluna, N) → olha N linhas para trás
        # A primeira compra de cada cliente terá null (não há anterior)
    ) \
    .withColumn(
        "variacao",                                   # Diferença: atual - anterior
        round(col("total_amount") - col("compra_anterior"), 2)
        # Positivo = gastou mais | Negativo = gastou menos
    )

# --- Mostrar resultado (apenas onde há comparação possível) ---
print("📈 TENDÊNCIA DE COMPRAS (comparação com compra anterior):")
print("   variacao > 0 → cliente gastou MAIS que na compra anterior")
print("   variacao < 0 → cliente gastou MENOS")
print()
df_tendencia \
    .filter(col("compra_anterior").isNotNull()) \
    .show(10, truncate=False)

### 4.3 Running Total: Soma Acumulada por Cliente

**"Quanto cada cliente já gastou ao longo do tempo?"**

Running total (soma acumulada) usa `rowsBetween` para definir a janela:  
- `unboundedPreceding` → desde o início (primeira compra do cliente)  
- `currentRow` → até a linha atual

Resultado: cada linha mostra o total acumulado até aquele momento.

In [ ]:
# =============================================================================
# PASSO 4.3: Soma acumulada (running total) por cliente
# =============================================================================

# Janela: do início da partição até a linha atual
# Cada cliente tem sua própria "janela" de soma
window_acumulada = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date") \
    .rowsBetween(
        Window.unboundedPreceding,  # Desde a primeira compra do cliente
        Window.currentRow            # Até esta compra (inclusive)
    )

# Aplicar sum() sobre a janela acumulada
df_acumulado = df_completo \
    .select("customer_id", "order_date", "total_amount") \
    .withColumn(
        "total_acumulado",                                 # Nova coluna
        round(sum("total_amount").over(window_acumulada), 2)  # Soma acumulada
    )

# --- Mostrar evolução de um cliente específico ---
# Pegamos o primeiro customer_id como exemplo
primeiro_cliente = df_acumulado.select("customer_id").first()[0]

print(f"📊 SOMA ACUMULADA — Cliente: {primeiro_cliente}")
print("   Observe como total_acumulado cresce a cada compra:")
print()
df_acumulado \
    .filter(col("customer_id") == primeiro_cliente) \
    .orderBy("order_date") \
    .show(10, truncate=False)

---

## 5. UDFs — User Defined Functions

### O que é uma UDF?

Uma **UDF** (User Defined Function) permite aplicar lógica Python customizada  
a cada registro do DataFrame. É útil quando as funções built-in do Spark não resolvem.

### Cuidado: UDFs são LENTAS!

| Abordagem | Performance | Por quê |
|-----------|-------------|--------|
| Built-in (`when/otherwise`) | ⚡ Máxima | Executado nativamente na JVM |
| UDF Python | 🐌 15x mais lenta | Serializa cada registro JVM → Python → JVM |

**Regra:** Sempre tente resolver com funções built-in primeiro. UDF é último recurso.

### 5.1 UDF para Classificar Ticket

A Ana quer segmentar pedidos por faixa de valor:  
- < R$ 50 → Baixo  
- R$ 50-199 → Médio  
- R$ 200-499 → Alto  
- ≥ R$ 500 → Premium

In [ ]:
# =============================================================================
# PASSO 5.1: UDF — Classificação de ticket (abordagem LENTA)
# =============================================================================

from pyspark.sql.functions import udf        # Decorador para criar UDFs
from pyspark.sql.types import StringType     # Tipo de retorno da UDF
import time                                   # Para medir performance

# Definir a UDF com o decorador @udf
# returnType=StringType() → a função retorna uma string
@udf(returnType=StringType())
def classificar_ticket(valor):
    """Classifica o valor do pedido em faixas de ticket.
    
    ATENÇÃO: Esta função é executada para CADA REGISTRO individualmente.
    Com 1M de registros = 1M de chamadas Python (LENTO!).
    """
    if valor is None:
        return "Indefinido"
    elif valor < 50:
        return "Baixo"
    elif valor < 200:
        return "Medio"
    elif valor < 500:
        return "Alto"
    else:
        return "Premium"

# Aplicar a UDF ao DataFrame
start = time.time()
df_com_udf = df_completo.withColumn(
    "faixa_ticket",                           # Nova coluna
    classificar_ticket(col("total_amount"))    # Aplica UDF em cada linha
)

# Forçar execução (UDF é lazy — só executa com uma ação)
resultado_udf = df_com_udf.groupBy("faixa_ticket") \
    .agg(
        count("*").alias("qtd_pedidos"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("qtd_pedidos")) \
    .collect()  # collect() traz resultado para o Driver (ação)
tempo_udf = time.time() - start

print("📊 DISTRIBUIÇÃO POR FAIXA DE TICKET (via UDF):")
print(f"⏱️  Tempo de execução: {tempo_udf:.2f}s")
print()
df_com_udf.groupBy("faixa_ticket") \
    .agg(count("*").alias("qtd"), round(avg("total_amount"), 2).alias("ticket_medio")) \
    .orderBy(desc("qtd")) \
    .show()

### 5.2 Alternativa RÁPIDA: when/otherwise (sem UDF)

A mesma lógica usando funções nativas do Spark — **15-30x mais rápido**  
porque executa diretamente na JVM sem serializar para Python.

In [ ]:
# =============================================================================
# PASSO 5.2: Mesma classificação SEM UDF (when/otherwise) — RÁPIDO!
# =============================================================================

from pyspark.sql.functions import when  # Equivalente a IF/ELSE em SQL

# when/otherwise: executado nativamente na JVM (sem ida para Python)
# Mesma lógica, performance 15x melhor
start = time.time()
df_nativo = df_completo.withColumn(
    "faixa_ticket",
    when(col("total_amount") < 50, "Baixo")        # Se < 50 → Baixo
    .when(col("total_amount") < 200, "Medio")      # Senão se < 200 → Medio
    .when(col("total_amount") < 500, "Alto")       # Senão se < 500 → Alto
    .otherwise("Premium")                           # Senão → Premium
)

# Forçar execução para comparar tempo
resultado_nativo = df_nativo.groupBy("faixa_ticket") \
    .agg(count("*").alias("qtd")) \
    .orderBy(desc("qtd")) \
    .collect()
tempo_nativo = time.time() - start

print("📊 DISTRIBUIÇÃO POR FAIXA (via when/otherwise nativo):")
print(f"⏱️  Tempo de execução: {tempo_nativo:.2f}s")
print()
df_nativo.groupBy("faixa_ticket") \
    .agg(count("*").alias("qtd")) \
    .orderBy(desc("qtd")) \
    .show()

# --- Comparativo ---
print("═" * 50)
print("COMPARATIVO DE PERFORMANCE:")
print(f"   UDF Python:        {tempo_udf:.2f}s")
print(f"   when/otherwise:    {tempo_nativo:.2f}s")
if tempo_udf > 0:
    print(f"   Speedup:           {tempo_udf/tempo_nativo:.1f}x mais rápido sem UDF!")
print("═" * 50)
print()
print("💡 CONCLUSÃO: Sempre prefira when/otherwise sobre UDFs!")
print("   UDFs só quando a lógica for impossível de expressar com built-ins.")

---

## 6. Análise Final — Relatório Consolidado para a Black Friday

Agora que temos o DataFrame completo (vendas + clientes + categorias),  
vamos gerar os relatórios que a Ana pediu.

### Cache: evitar recomputação

Como vamos usar `df_completo` múltiplas vezes, fazemos `cache()`.  
Sem cache, cada ação (show, count) recalcularia TODO o pipeline (joins + derivações).

In [ ]:
# =============================================================================
# PASSO 6: Relatórios finais (com cache para performance)
# =============================================================================

# cache() armazena o DataFrame na memória dos executors
# IMPORTANTE: cache é lazy — só materializa na próxima ação!
df_completo.cache()
df_completo.count()  # Força materialização do cache

print("✅ DataFrame completo cacheado na memória!")
print("   Próximas consultas serão instantâneas.")
print()

# --- RELATÓRIO 1: Faturamento por Categoria ---
print("═" * 60)
print("RELATÓRIO 1: FATURAMENTO POR CATEGORIA")
print("═" * 60)
df_completo.groupBy("category_name") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento"),
        count("order_id").alias("pedidos"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento")) \
    .show(truncate=False)

# --- RELATÓRIO 2: Faturamento por Estado ---
print("═" * 60)
print("RELATÓRIO 2: TOP 10 ESTADOS POR FATURAMENTO")
print("═" * 60)
df_completo.groupBy("shipping_state") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento"),
        count("order_id").alias("pedidos"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento")) \
    .show(10)

# --- RELATÓRIO 3: Vendas por Segmento de Cliente ---
print("═" * 60)
print("RELATÓRIO 3: VENDAS POR SEGMENTO DE CLIENTE")
print("═" * 60)
df_completo.groupBy("segment") \
    .agg(
        count("order_id").alias("pedidos"),
        round(sum("total_amount"), 2).alias("faturamento"),
        round(avg("total_amount"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento")) \
    .show()

# --- Liberar memória ---
# unpersist() remove o DataFrame do cache (boa prática em notebooks longos)
df_completo.unpersist()
print("\n🧹 Cache liberado.")

---

## 7. Encerramento

Sempre feche a SparkSession ao finalizar para liberar recursos.

In [ ]:
# =============================================================================
# PASSO 7: Encerrar SparkSession
# =============================================================================

spark.stop()
print("✅ SparkSession encerrada. Recursos liberados.")

---

# DESAFIO

## Análise de Retenção e Risco de Churn

Você é o Carlos Mendes. A equipe de marketing quer identificar clientes em risco  
de abandonar a plataforma (**churn**) antes da Black Friday, para enviar cupons de reativação.

---

### Requisitos:

1. **Identifique clientes recorrentes** — clientes com mais de 1 compra  
   *Dica:* use `count().over(Window.partitionBy("customer_id"))`

2. **Calcule o tempo entre compras** — dias entre compras consecutivas  
   *Dica:* use `lag("order_date", 1)` + `datediff()`

3. **Classifique o risco de churn:**  
   - Última compra > 90 dias atrás → **Alto risco**  
   - Última compra 30-90 dias → **Médio risco**  
   - Última compra < 30 dias → **Baixo risco**  
   *Dica:* use `datediff(current_date(), max("order_date"))` + `when/otherwise`

4. **Gere um relatório** com: total de clientes por faixa de risco e ticket médio por faixa

5. **Use `left_anti` join** para encontrar clientes que nunca compraram (órfãos)  
   *Dica:* `df_clientes.join(df_vendas, on="customer_id", how="left_anti")`

---

### Estrutura Sugerida:

```python
from pyspark.sql.functions import datediff, current_date, max as spark_max

# 1. Calcular última compra e total gasto por cliente
df_resumo_cliente = df_completo.groupBy("customer_id").agg(
    spark_max("order_date").alias("ultima_compra"),
    sum("total_amount").alias("total_gasto"),
    count("order_id").alias("num_compras")
)

# 2. Calcular dias desde última compra
df_resumo_cliente = df_resumo_cliente.withColumn(
    "dias_desde_ultima",
    datediff(current_date(), col("ultima_compra"))
)

# 3. Classificar risco com when/otherwise
# ... seu código aqui ...

# 4. Relatório por faixa de risco
# ... seu código aqui ...

# 5. Clientes órfãos (left_anti)
# ... seu código aqui ...
```

---

### Bônus:
- Compare o `explain()` de um join normal vs left_anti
- Use `ntile(4)` para dividir clientes em quartis por faturamento
- Calcule a média móvel de 3 compras com `rowsBetween(-2, Window.currentRow)`

---

**Boa sorte!** Análise de churn é um caso real em qualquer empresa de e-commerce.

In [ ]:
# ===========================================================
# SEU CÓDIGO DO DESAFIO AQUI
# ===========================================================

# Recrie a SparkSession (foi encerrada no passo 7)
# spark = SparkSession.builder.appName("Desafio-Churn").master("local[*]").getOrCreate()

# Seu código abaixo:

